In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
# Initialize Spark with Kafka and Postgres packages
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
# 1. Load Static User Data (From CSV or Postgres)
# If using CSV:
users_df = spark.read.csv("data/user_table.csv", header=True,
inferSchema=True)
# users_df = spark.read \
#  .format("jdbc") \
#  .option("url", "jdbc:postgresql://postgres:5432/userdb") \
#  .option("dbtable", "users") \
#  .option("user", "admin") \
#  .option("password", "password") \
#  .load()
# 2. Read Streaming Data from Kafka
tx_schema = StructType([
 StructField("tx_id", IntegerType(), True),
 StructField("userId", IntegerType(), True),
 StructField("amount", DoubleType(), True),
 StructField("timestamp", DoubleType(), True)
])
kafka_stream = spark.readStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("subscribe", "fraud-detection") \
 .load()
# 3. Parse and Filter Data
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),
tx_schema).alias("tx")).select("tx.*")
# Fraud Logic: Amount > 10,000
fraud_stream = parsed_stream.filter(col("amount") > 5000.00)
# 4. Enrich Stream with User Details (Stream-Static Join)
enriched_fraud = fraud_stream.join(users_df, "userId")
# 5. Format for output Kafka topic
output_stream = enriched_fraud \
 .withColumn("value", to_json(struct("*")).cast("string")) \
 .select("value")
# 6. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification") \
 .option("checkpointLocation", "/tmp/checkpoints") \
 .start()
query.awaitTermination()

26/06/19 06:07:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:07:01 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                